# Laya vs FinBERT: financial news sentiment (experiment A)

**Kaggle settings before running:** Accelerator = GPU T4, Internet = On.
Optional: add `news_items.csv` as a dataset (Add Input -> Upload) to run section 5 on our own headlines.

Questions this notebook answers:
1. How good is Laya **zero-shot** (no training) vs FinBERT on labeled financial news (`zeroshot/twitter-financial-news-sentiment`, MIT)?
2. How fast is each model (latency per item)?
3. What does Laya say about our own headlines? (writes a file to hand-label)

Untested draft: the Laya output structure is taken from its model card. Section 3 prints one raw result so you can fix the parsing if it differs.

In [ ]:
import os
os.environ["USE_TF"] = "0"   # model card: avoids a TensorFlow deadlock when loading Laya
%pip install -q laya datasets scikit-learn

In [ ]:
import time, glob
import numpy as np, pandas as pd, torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

N_EVAL = None          # set e.g. 500 for a quick smoke test
LABELS = ["bearish", "bullish", "neutral"]   # dataset ints 0,1,2
DEVICE = 0 if torch.cuda.is_available() else -1
print("GPU:", torch.cuda.get_device_name(0) if DEVICE == 0 else "none (will be slow)")

ds = load_dataset("zeroshot/twitter-financial-news-sentiment", split="validation")
texts, y = list(ds["text"]), np.array(ds["label"])
if N_EVAL:
    texts, y = texts[:N_EVAL], y[:N_EVAL]
print(len(texts), "labeled examples; label counts:", dict(zip(LABELS, np.bincount(y))))

results = {}   # name -> dict(pred, secs_per_item)

def score(name, pred, secs):
    results[name] = {"pred": np.array(pred), "ms_per_item": 1000 * secs / len(pred)}
    print(f"{name}: acc={accuracy_score(y, pred):.3f}  macro-F1={f1_score(y, pred, average='macro'):.3f}  {1000*secs/len(pred):.1f} ms/item (batch/loop total {secs:.1f}s)")

## 1. FinBERT baseline (batched)

In [ ]:
from transformers import pipeline
finbert = pipeline("text-classification", model="ProsusAI/finbert", device=DEVICE, truncation=True, batch_size=32)
fb_map = {"negative": 0, "positive": 1, "neutral": 2}
finbert(texts[:8])  # warm-up
t = time.time()
out = finbert(texts)
score("finbert", [fb_map[o["label"].lower()] for o in out], time.time() - t)

## 2. Laya zero-shot

We ask ONE `choice` question. Laya answers one item at a time here (the model card shows no batch call), so the ms/item below is a loop, not batched throughput.

In [ ]:
import laya

SENTIMENT_Q = {"sentiment": {
    "type": "choice",
    "instructions": "What is the likely effect of this financial news on the stock price?",
    "criteria": {
        "bearish": "negative for the stock price",
        "bullish": "positive for the stock price",
        "neutral": "no clear effect, or purely informational",
    },
}}
choice_to_int = {"bearish": 0, "bullish": 1, "neutral": 2}

def run_laya(name, **load_kwargs):
    agent = laya.load("convaiinnovations/laya", **load_kwargs)
    first = agent.predict(texts[0], SENTIMENT_Q)
    print(name, "raw result for first example:\n", first)      # <- check the structure here
    agent.predict(texts[1], SENTIMENT_Q)                         # warm-up
    preds, t = [], time.time()
    for s in texts:
        r = agent.predict(s, SENTIMENT_Q)
        preds.append(choice_to_int[r["answers"]["sentiment"]["choice"]])
    score(name, preds, time.time() - t)
    return agent

agent_en = run_laya("laya_english_zeroshot")

In [ ]:
agent_td = run_laya("laya_typed_decisions_zeroshot", subfolder="typed-decisions")

## 3. Comparison

In [ ]:
rows = []
for name, r in results.items():
    rows.append({"model": name,
                 "accuracy": accuracy_score(y, r["pred"]),
                 "macro_F1": f1_score(y, r["pred"], average="macro"),
                 "ms_per_item": r["ms_per_item"]})
print(pd.DataFrame(rows).round(3).to_string(index=False))
for name, r in results.items():
    print("\n", name, "(rows = true, cols = predicted; order:", LABELS, ")")
    print(confusion_matrix(y, r["pred"]))

## 4. Reading the result (fill in after running)

- If Laya zero-shot is near chance (~0.33-0.40 accuracy), that matches the model card: it needs fine-tuning. FinBERT is then the baseline to beat.
- Compare ms/item with care: FinBERT is batched, Laya is a one-at-a-time loop.
- This is tweet-style financial news, not our Google News headlines. Section 5 checks our own data.

## 5. Our own headlines (needs `news_items.csv` as an input dataset)

In [ ]:
paths = glob.glob("/kaggle/input/**/news_items.csv", recursive=True)
assert paths, "Add news_items.csv as a dataset input first"
news = pd.read_csv(paths[0])
sample = news.sample(min(1000, len(news)), random_state=0).reset_index(drop=True)
print(len(news), "headlines total; using", len(sample))

# sentiment: FinBERT (fast) + Laya english
inv = {v: k for k, v in fb_map.items()}
sample["finbert"] = [LABELS[fb_map[o["label"].lower()]] for o in finbert(list(sample.title), batch_size=32)]
sample["laya_sentiment"] = [agent_en.predict(s, SENTIMENT_Q)["answers"]["sentiment"]["choice"] for s in sample.title]

# noise filter: "is this a concrete market-relevant event or filler?"
NOISE_Q = {"kind": {
    "type": "choice",
    "instructions": "Is this headline a concrete company event, or generic filler?",
    "criteria": {
        "event": "reports a concrete event: earnings, guidance, a deal, a rating change, a product or legal or regulatory action involving the company",
        "filler": "generic explainer, stock-quote page, promotion, or fund-holding boilerplate",
    },
}}
sample["laya_kind"] = [agent_en.predict(s, NOISE_Q)["answers"]["kind"]["choice"] for s in sample.title]

print("\nFinBERT vs Laya sentiment agreement:", (sample.finbert == sample.laya_sentiment).mean().round(3))
print("\nLaya 'filler' rate vs keyword-rule flag:")
print(pd.crosstab(sample.rule_noise, sample.laya_kind))
sample.to_csv("predictions.csv", index=False)

## 6. Gold set to hand-label

Open `to_label.csv`, fill `gold_sentiment` (bearish/bullish/neutral) and `gold_kind` (event/filler) yourself for ~200 rows. Without this we cannot say which model is right on OUR headlines.

In [ ]:
to_label = sample.sample(200, random_state=1)[["ticker", "title", "source", "published_at"]].copy()
to_label["gold_sentiment"] = ""
to_label["gold_kind"] = ""
to_label.to_csv("to_label.csv", index=False)
print("Saved predictions.csv and to_label.csv (download from the Output tab)")